# **Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Ali Yassine & Lorenzo Vaiani

**Credits:** Moreno La Quatra

**Practice 3:** Named Entities Recognition & Intent Detection

## Named Entities Recognition (NER)

The Named Entity Recognition task aims at identifying and classifying named entities in a text. Named entities are real-world objects such as persons, locations, organizations, etc. The task takes as input a sentence and determines the boundaries of the named entities and their type.

For example, given the sentence:

```
I went to Paris last week.
```

the task is to identify the named entity `Paris` as a location.

Hereafter an illustration of the NER task:

![NER example](https://raw.githubusercontent.com/aliyassine26/DNLP/main/2026-2027/Practice_3/ner_illustration.png)

In the first part of this practice, you will:
- explore the NER task using pre-trained models available on spaCy and HuggingFace
- evaluate the performance of a spaCy NER model on a custom dataset
- evaluate the performance of a HuggingFace NER model on a custom dataset

NB: The library used to evaluate the performance of the models is `seqeval`, which is a library for evaluating sequence labeling tasks.

### **Question 1: Data preparation**

The first step is to prepare the data. In this practice, you will use the WikiGold dataset [1][2], which is a collection of annotated sentences from Wikipedia. The dataset is available in [CONLL](https://simpletransformers.ai/docs/ner-data-formats/#text-file-in-conll-format) format. The dataset is available [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt).

**Please, read carefully the following instructions before starting to work on the practice.**

You need to extract clean sentences (no annotation) and, for each sentence, the corresponding annotations. The dataset has the following format:

- `sentences`: list of sentences
- `annotations`: list of list of entities (both string and class information). E.g., `[[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')], [('Osc-Dis', 'MISC'), ('Introduction 010', 'MISC'), ('Come', 'MISC')], ...]`. You can remove I- prefix because the data collection does not actually contain valuable prefixes.

---


[1] Balasuriya, Dominic, et al. "Named entity recognition in wikipedia."
    Proceedings of the 2009 Workshop on The People's Web Meets NLP: Collaboratively Constructed Semantic Resources. Association for Computational Linguistics, 2009.

[2] Nothman, Joel, et al. "Learning multilingual named entity recognition
    from Wikipedia." Artificial Intelligence 194 (2013): 151-175

---

The following cell downloads the dataset on Google Colab.

In [ ]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt

--2026-10-29 15:51:36--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 318530 (311K) [text/plain]
Saving to: ‘wikigold.conll.txt’

wikigold.conll.txt  100%[===================>] 311.06K  --.-KB/s    in 0.01s   

2026-10-29 15:51:36 (22.6 MB/s) - ‘wikigold.conll.txt’ saved [318530/318530]



To avoid spending too much time on data processing, the following cells prepare the dataset for you.
After running the cell, you will have the following variables:
- `sentences_with_labels`: a list of tokens with their corresponding labels
- `sentences`: a list of the sentences in the dataset
- `labels`: a list of lists of labels. Each element in the outer list corresponds to a list of labels for a sentence in the dataset.

Hereafter an example of the provided data:

```python
sentences_with_labels[0] = [
    ['010', 'I-MISC'],
    ['is', 'O'],
    ['the', 'O'],
    ['tenth', 'O'],
    ['album', 'O'],
    ['from', 'O'],
    ['Japanese', 'I-MISC'],
    ['Punk', 'O'],
    ['Techno', 'O'],
    ['band', 'O'],
    ['The', 'I-ORG'],
    ['Mad', 'I-ORG'],
    ['Capsule', 'I-ORG'],
    ['Markets', 'I-ORG'],
    ['.', 'O']
]

sentences[0] = [
    '010 is the tenth album from Japanese Punk Techno band The Mad Capsule Markets .'
]

labels[0] = [
    ('010', 'MISC'),
    ('Japanese', 'MISC'),
    ('The Mad Capsule Markets', 'ORG')
]
```

Please, note that the labels are not in IOB format. You can ignore the I- prefix because the data collection does not actually contain valuable prefixes.
Get familar with the data by printing the first 10 sentences and their corresponding labels. Which are the labels in the dataset?

In [ ]:
%%capture
! pip install datasets
! pip install transformers
! pip install spacy
! python -m spacy download en_core_web_sm

In [ ]:
def split_text_label(filename):
    f = open(filename)
    split_labeled_text = []
    sentence = []
    for line in f:
        if len(line)==0 or line.startswith('-DOCSTART') or line[0]=="\n":
             if len(sentence) > 0:
                 split_labeled_text.append(sentence)
                 sentence = []
             continue
        splits = line.split(' ')
        sentence.append([splits[0],splits[-1].rstrip("\n")])
    if len(sentence) > 0:
        split_labeled_text.append(sentence)
        sentence = []
    return split_labeled_text
sentences_with_labels = split_text_label("wikigold.conll.txt")

In [ ]:
print(sentences_with_labels[0])

[['010', 'I-MISC'], ['is', 'O'], ['the', 'O'], ['tenth', 'O'], ['album', 'O'], ['from', 'O'], ['Japanese', 'I-MISC'], ['Punk', 'O'], ['Techno', 'O'], ['band', 'O'], ['The', 'I-ORG'], ['Mad', 'I-ORG'], ['Capsule', 'I-ORG'], ['Markets', 'I-ORG'], ['.', 'O']]


In [ ]:
sentences = []

for sent_list in sentences_with_labels:
    sentence = [s[0] for s in sent_list]
    sentence = " ".join(sentence)
    sentences.append(sentence)

In [ ]:
print (sentences[0])

010 is the tenth album from Japanese Punk Techno band The Mad Capsule Markets .


In [ ]:
labels = []
overall_labels = []
for sent_list in sentences_with_labels:
    current_labels = []
    prev = "O"
    current_entity = ""
    for w, l in sent_list:
        overall_labels.append(l)
        if l != "O" and prev != "O":
            if l == prev:
                # continue entity
                current_entity += w + " "
            else:
                # end prev and start a new one
                current_labels.append((current_entity.strip(), prev.split("-")[1]))
                current_entity = w + " "
        elif l == "O" and  prev != "O":
            # end prev
            current_labels.append((current_entity.strip(), prev.split("-")[1]))
            current_entity = ""
        elif l != "O" and prev == "O":
            # start new
            current_entity = w + " "

        prev = l
    labels.append(current_labels)

print (labels)
overall_labels = list(set(overall_labels))
overall_labels = [o for o in overall_labels if o != "O"]
overall_labels = [o.split("-")[1] for o in overall_labels]
print (overall_labels)

[[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')], [('Osc-Dis', 'MISC'), ('Introduction 010', 'MISC'), ('Come', 'MISC')], [('Kojima Minoru', 'PER'), ('Good Day', 'MISC'), ('Wardanceis', 'MISC'), ('UK', 'LOC'), ('Killing Joke', 'ORG')], [('XXX can of This', 'MISC')], [('Cannabis', 'MISC'), ('Cannabis', 'MISC'), ('P.O.P', 'MISC'), ('HUMANITY', 'MISC')], [('UK', 'LOC'), ('OSC-DIS', 'MISC')], [('139th', 'ORG'), ('Camp Howe', 'LOC'), ('Pittsburgh', 'LOC')], [('Frederick H. Collier', 'PER')], [('Second Battle of Bull Run', 'MISC'), ("Howe 's Brigade", 'ORG'), ("Couch 's Division", 'ORG'), ('IV Corps', 'ORG'), ('Army of the Potomac', 'ORG'), ("De Trobriand 's 55th New York", 'ORG'), ('Gardes Lafayette', 'ORG')], [('62d NY', 'ORG'), ('93d PVI', 'ORG'), ('98th PVI', 'ORG'), ('102d PVI', 'ORG')], [("Couch 's Division", 'ORG'), ('Army', 'ORG'), ('Potomac', 'LOC')], [('139th', 'ORG'), ('Poolesville', 'LOC'), ('Sandy Hook', 'LOC'), ('Maryland', 'LOC'), ('Battle of Antieta

In [ ]:
print (labels[0])

[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')]


### **Question 2: Inference with spaCy for entity recognition**

**spaCy** is a free, open-source **library for advanced Natural Language Processing** in Python. It features NER models for different languages including English.
The models are available [here](https://spacy.io/models).

For this question you are asked to instantiate a spaCy model for English and **perform inference on the sentences in the dataset**. The English model contains a superset of the labels in the dataset.

For this reason, you need to map the labels that are not in the dataset to the `MISC` label.

You are expected to generate an output similar to the following:

```python

[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')]

```

Please pay attention to the token attributes (you can find more information [here](https://spacy.io/api/token#attributes)) and the entity attributes (you can find more information [here](https://spacy.io/api/entityrecognizer)).

The following cell instantiates a spaCy model for English.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
l = []
for i in range(len(labels)):
  for c in labels[i]:
    if c[1] not in l:
      l.append(c[1])

l

['MISC', 'ORG', 'PER', 'LOC']

In [ ]:
doc = nlp("My name is Luca, I work at Apple, in the main building near Menlo Park")
print([(ent.text, ent.label_) for  ent in doc.ents])
print([(ent.text, ent.label_) if ent.label_ in l else (ent.text,'MISC') for  ent in doc.ents])

def to_gt_format(a_list):
  new_list = []
  for ele in a_list:
    label = ""
    if ele.label_ == 'PERSON':
      label = 'PER'
    elif ele.label_ == 'GPE':
      label = 'LOC'
    elif ele.label_ == 'ORG':
      label = 'ORG'
    else:
      label = 'MISC'
    new_list.append((ele.text, label))
  return new_list

print()
print([(ent.text, ent.label_) for  ent in doc.ents])
print( to_gt_format(doc.ents) )

[('Luca', 'PERSON'), ('Apple', 'ORG'), ('Menlo Park', 'GPE')]
[('Luca', 'MISC'), ('Apple', 'ORG'), ('Menlo Park', 'MISC')]

[('Luca', 'PERSON'), ('Apple', 'ORG'), ('Menlo Park', 'GPE')]
[('Luca', 'PER'), ('Apple', 'ORG'), ('Menlo Park', 'LOC')]


In [ ]:
spacy_labels = [to_gt_format(nlp_sentence.ents) for nlp_sentence in [nlp(sentence) for sentence in sentences]]

In [ ]:
spacy_labels[1]

[('Osc-Dis', 'ORG'), ('Introduction 010', 'MISC')]

In [ ]:
labels[1]

[('Osc-Dis', 'MISC'), ('Introduction 010', 'MISC'), ('Come', 'MISC')]

In [ ]:
sentences[1]

'This album proved to be more commercial and more techno-based than Osc-Dis , with heavily synthesized songs like Introduction 010 and Come .'

### **Question 3: Compute metrics for evaluating NER**

The output of NER models consists of a set of named entities. To evaluate the performance of a model, we need to compare the predicted named entities with the ground truth.

For this question, you need to use [`eval4ner`](https://github.com/cyk1337/eval4ner) package to evaluate the performance of the model.

In [ ]:
%%capture
! pip install eval4ner

### 1. Strict
* **What it means:** 100% Exact Match.
* Both the text boundaries (start/end positions) and the category label (e.g., `PER`, `LOC`, `ORG`) must match the ground truth perfectly.
* **Example:** Standard is `PER: "John Smith"`.
  * Prediction `PER: "John Smith"` → Correct
  * Prediction `ORG: "John Smith"` → Wrong

### 2. Exact Boundary Matching *(Docs meant: "regardless of type")*
* **What it means:** Right Span, Ignore Category.
* The model found the exact start and end of the entity text, but we don't care if it labeled the category correctly.
* **Example:** Standard is `PER: "John Smith"`.
  * Prediction `ORG: "John Smith"` → Correct (the boundary "John Smith" matches perfectly).

### 3. Partial Boundary Matching *(Docs meant: "regardless of type")*
* **What it means:** Partial Overlap, Ignore Category.
* The model got some of the words right, but missed the exact boundaries (e.g., grabbed extra words or missed a word). It gets partial credit (0.5), regardless of the category label.
* **Example:** Standard is `LOC: "New York City"`.
  * Prediction `LOC: "New York"` or `ORG: "York City"` → Partial Credit (overlaps, even if the label is wrong).

### 4. Type Matching
* **What it means:** Right Category, Ignore Exact Span.
* As long as there is at least some overlap in the text boundary, did the model assign the right entity label (`PER`, `LOC`, etc.)?
* **Example:** Standard is `LOC: "New York City"`.
  * Prediction `LOC: "New York"` → Correct (assigned `LOC` correctly, despite missing "City").
  * Prediction `ORG: "New York"` → Wrong (label `ORG` is incorrect).

In [ ]:
import eval4ner.muc as muc
muc.evaluate_all(spacy_labels, labels, sentences, verbose=True)


Output streaming troncato alle ultime 5000 righe.
{'exact': {'actual': 6,
           'correct': 4,
           'f1_score': 0.5333333333333333,
           'incorrect': 0,
           'missed': 5,
           'partial': 0,
           'possible': 9,
           'precision': 0.6666666666666666,
           'recall': 0.4444444444444444,
           'spurius': 2},
 'partial': {'actual': 6,
             'correct': 4,
             'f1_score': 0.5333333333333333,
             'incorrect': 0,
             'missed': 5,
             'partial': 0,
             'possible': 9,
             'precision': 0.6666666666666666,
             'recall': 0.4444444444444444,
             'spurius': 2},
 'strict': {'actual': 6,
            'correct': 4,
            'f1_score': 0.5333333333333333,
            'incorrect': 0,
            'missed': 5,
            'partial': 0,
            'possible': 9,
            'precision': 0.6666666666666666,
            'recall': 0.4444444444444444,
            'spurius': 2},
 'typ

{'strict': {'precision': 0.3706798560822472,
  'recall': 0.2762306198993752,
  'f1_score': 0.30658460065979953,
  'count': 1696},
 'exact': {'precision': 0.5095382387213921,
  'recall': 0.38151181989270816,
  'f1_score': 0.4216814421299521,
  'count': 1696},
 'partial': {'precision': 0.5095382387213921,
  'recall': 0.38151181989270816,
  'f1_score': 0.4216814421299521,
  'count': 1696},
 'type': {'precision': 0.3706798560822472,
  'recall': 0.2762306198993752,
  'f1_score': 0.30658460065979953,
  'count': 1696}}

### MUC-5 Evaluation Metric Framework

MUC-5 evaluates Named Entity Recognition (NER) models by categorizing predictions into 5 possible outcome classes:

* **COR (Correct):** Entity boundary and/or type match the ground truth.
* **INC (Incorrect):** Entity detected, but boundary or type is wrong.
* **PAR (Partial):** Partial boundary overlap (receives half credit).
* **MIS (Missed):** Entity present in ground truth but missed by the model (False Negative).
* **SPU (Spurious):** Model predicted an entity where none exists in ground truth (False Positive).

---

#### 1. Total Entity Counts

* **POS (Possible / Gold Standard Total):**
  $$\text{POS} = \text{COR} + \text{INC} + \text{PAR} + \text{MIS} = TP + FN$$

* **ACT (Actual / Predictions Total):**
  $$\text{ACT} = \text{COR} + \text{INC} + \text{PAR} + \text{SPU} = TP + FP$$

---

#### 2. Metric Calculations

##### Strict & Exact Matching (No partial credit)
$$\text{Precision} = \frac{\text{COR}}{\text{ACT}}$$

$$\text{Recall} = \frac{\text{COR}}{\text{POS}}$$

##### Partial & Type Matching (0.5 weight for partial overlaps)
$$\text{Precision} = \frac{\text{COR} + 0.5 \times \text{PAR}}{\text{ACT}}$$

$$\text{Recall} = \frac{\text{COR} + 0.5 \times \text{PAR}}{\text{POS}}$$

---

#### 3. F-Score Measures

* **General $F_\alpha$ Score:**
  $$F_\alpha = \frac{(\alpha^2 + 1) \cdot P \cdot R}{\alpha^2 P + R}$$

* **Balanced $F_1$ Score ($\alpha = 1$):**
  $$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

### **Question 4: Inference with transformers pipeline**

Transformer-based models can be fine-tuned for token-level classification. The task is to classify each token in a sentence and assign it to a class.
The NER task is a token-level classification task and the models can be used for performing inference on the sentences in the dataset.

You can use the pipeline available on the HuggingFace [transformers library](https://huggingface.co/docs/transformers/main_classes/pipelines). The pipeline allows to perform inference on a list of sentences.

Evaluate the **standard** model using the pipeline (`pipe = pipeline("ner")`). Check the documentation here: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.TokenClassificationPipeline


A few notes about the question (**read carefully**):
1. The output of the pipeline differs with respect to spaCy. Please be sure to process data correctly before running evaluation.
2. `ignore_labels` parameter could be used to exclude labels from the prediction.
3. `##` symbol is used when a token is a continuation of a previous one (Poli + ##TO). You may need to check this specific case to merge the tokens correctly.
4. Use `seqeval` to evaluate the performance of the model.

## More on pipeline

The pipelines are a great and easy way to use models for inference. These pipelines are objects that abstract most of the complex code from the library, offering a simple API dedicated to several tasks, including Named Entity Recognition, Masked Language Modeling, Sentiment Analysis, Feature Extraction and Question Answering. See the task list for examples of use.

There are two categories of pipeline abstractions to be aware about:

- The pipeline() which is the most powerful object encapsulating all other pipelines.
- Task-specific pipelines are available for audio, computer vision, natural language processing, and multimodal tasks.

The pipeline abstraction
The pipeline abstraction is a wrapper around all the other available pipelines. It is instantiated as any other pipeline but can provide additional quality of life.

Simple call on one item:

In [ ]:
%%capture
! pip install transformers
! pip install datasets
! pip install accelerate

In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification")
pipe("This restaurant is awesome")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998743534088135}]

In [ ]:
pipe = pipeline("ner", ignore_labels=[]) ## I set ignore_labels parameter to [] to retain 0 label in prediction output

from datasets import load_dataset

dataset = load_dataset("milistu/Wikigold-NER-conll")


[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
label_name = {
    0: "O",
    1: "B-LOC",
    2: "I-LOC",
    3: "B-MISC",
    4: "I-MISC",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-PER",
    8: "I-PER",
}

gt = []
for val in dataset['train']:
  an_element = []
  for word, label in zip(val['words'],val['ner_tags']):
    an_element.append((word,label_name[label]))
  gt.append(an_element)

print(gt[0])

[('After', 'O'), ('the', 'O'), ('season', 'O'), (',', 'O'), ('the', 'O'), ('franchise', 'O'), ('was', 'O'), ('terminated', 'O'), ('and', 'O'), ('Art', 'I-PER'), ('Ross', 'I-PER'), (',', 'O'), ('the', 'O'), ('general', 'O'), ('manager', 'O'), ('of', 'O'), ('the', 'O'), ('Bruins', 'I-ORG'), (',', 'O'), ('selected', 'O'), ('him', 'O'), ('in', 'O'), ('the', 'O'), ('subsequent', 'O'), ('dispersal', 'O'), ('draft', 'O'), ('.', 'O')]


In [ ]:
sentences_with_labels[0]

[['010', 'I-MISC'],
 ['is', 'O'],
 ['the', 'O'],
 ['tenth', 'O'],
 ['album', 'O'],
 ['from', 'O'],
 ['Japanese', 'I-MISC'],
 ['Punk', 'O'],
 ['Techno', 'O'],
 ['band', 'O'],
 ['The', 'I-ORG'],
 ['Mad', 'I-ORG'],
 ['Capsule', 'I-ORG'],
 ['Markets', 'I-ORG'],
 ['.', 'O']]

In [ ]:
out1 = [ (val['word'], val['entity']) for val in pipe(sentences[0]) ]
print(out1)

[('01', 'O'), ('##0', 'O'), ('is', 'O'), ('the', 'O'), ('tenth', 'O'), ('album', 'O'), ('from', 'O'), ('Japanese', 'I-MISC'), ('Punk', 'I-MISC'), ('Tech', 'I-MISC'), ('##no', 'I-MISC'), ('band', 'O'), ('The', 'I-ORG'), ('Mad', 'I-ORG'), ('Cap', 'I-ORG'), ('##sul', 'I-ORG'), ('##e', 'I-ORG'), ('Markets', 'I-ORG'), ('.', 'O')]


In [ ]:
def join_words(words_list):
    new_list = []

    for word, label in words_list:
        if word.startswith("##") and new_list:
            # Pop the previous token, remove '##', merge, and re-append
            prev_word, prev_label = new_list.pop()
            new_list.append((prev_word + word[2:], prev_label))
        else:
            new_list.append((word, label))

    return new_list

In [ ]:
print(join_words(out1))

[('010', 'O'), ('is', 'O'), ('the', 'O'), ('tenth', 'O'), ('album', 'O'), ('from', 'O'), ('Japanese', 'I-MISC'), ('Punk', 'I-MISC'), ('Techno', 'I-MISC'), ('band', 'O'), ('The', 'I-ORG'), ('Mad', 'I-ORG'), ('Capsule', 'I-ORG'), ('Markets', 'I-ORG'), ('.', 'O')]


In [ ]:
ner_out = []
for sentence in sentences:
  ner_out.append( join_words([ (val['word'], val['entity']) for val in pipe(sentence) ]) )


In [ ]:
ner_out[0]

[('010', 'O'),
 ('is', 'O'),
 ('the', 'O'),
 ('tenth', 'O'),
 ('album', 'O'),
 ('from', 'O'),
 ('Japanese', 'I-MISC'),
 ('Punk', 'I-MISC'),
 ('Techno', 'I-MISC'),
 ('band', 'O'),
 ('The', 'I-ORG'),
 ('Mad', 'I-ORG'),
 ('Capsule', 'I-ORG'),
 ('Markets', 'I-ORG'),
 ('.', 'O')]

In [ ]:
sentences_with_labels[0]

[['010', 'I-MISC'],
 ['is', 'O'],
 ['the', 'O'],
 ['tenth', 'O'],
 ['album', 'O'],
 ['from', 'O'],
 ['Japanese', 'I-MISC'],
 ['Punk', 'O'],
 ['Techno', 'O'],
 ['band', 'O'],
 ['The', 'I-ORG'],
 ['Mad', 'I-ORG'],
 ['Capsule', 'I-ORG'],
 ['Markets', 'I-ORG'],
 ['.', 'O']]

In [ ]:
!pip install seqeval

In [ ]:
from seqeval.metrics import accuracy_score
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

In [ ]:
label_name = {
    0: "O",
    1: "B-LOC",
    2: "I-LOC",
    3: "B-MISC",
    4: "I-MISC",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-PER",
    8: "I-PER",
}

gt = []
for val in dataset['train']:
  an_element = []
  for v in val['ner_tags']:
    an_element.append(label_name[v])
  gt.append(an_element)
print(gt[0])



['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
dataset['train'][0]

{'id': '812',
 'words': ['After',
  'the',
  'season',
  ',',
  'the',
  'franchise',
  'was',
  'terminated',
  'and',
  'Art',
  'Ross',
  ',',
  'the',
  'general',
  'manager',
  'of',
  'the',
  'Bruins',
  ',',
  'selected',
  'him',
  'in',
  'the',
  'subsequent',
  'dispersal',
  'draft',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  8,
  8,
  0,
  0,
  0,
  0,
  0,
  0,
  6,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [ ]:
preds = []
for val1 in dataset['train']:
  an_ele = []
  sentence = " ".join(val1['words'])
  pred = join_words([(val['word'], val['entity']) for val in (pipe(sentence))])
  for val2 in pred:
    an_ele.append(val2[1])
  preds.append(an_ele)

In [ ]:
preds[0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [ ]:
gt[0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [ ]:
final_gt = []
final_preds = []
for val in zip(gt,preds):
  if (len(val[0])-len(val[1]))==0:
    final_gt.append(val[0])
    final_preds.append(val[1])


In [ ]:
len(gt)

1472

In [ ]:
len(final_gt)

867

In [ ]:
print("----------- Aligned outputs report  ("+str(len(final_gt)/len(gt)*100)+" % of total)--------------- \n\n "+classification_report(final_gt,final_preds))

----------- Aligned outputs report  (58.89945652173913 % of total)--------------- 

               precision    recall  f1-score   support

         LOC       0.84      0.89      0.86       501
        MISC       0.68      0.80      0.73       352
         ORG       0.71      0.73      0.72       373
         PER       0.93      0.95      0.94       383

   micro avg       0.79      0.85      0.82      1609
   macro avg       0.79      0.84      0.81      1609
weighted avg       0.80      0.85      0.82      1609



## Intent Detection

In data mining, intention mining or intent mining is the problem of determining a user's intention from logs of his/her behavior in interaction with a computer system, such as in search engines. Intent Detection is the identification and categorization of what a user online intended or wanted to find when they type or speak with a conversational agent (or a search engine).

![https://d33wubrfki0l68.cloudfront.net/32e2326762c75a0357ab1ae1976a60d4bbce724b/f4ac0/static/a5878ba6b0e4e77163dc07d07ecf2291/2b6c7/intent-classification-normal.png](https://d33wubrfki0l68.cloudfront.net/32e2326762c75a0357ab1ae1976a60d4bbce724b/f4ac0/static/a5878ba6b0e4e77163dc07d07ecf2291/2b6c7/intent-classification-normal.png)

In this section, you will use the ATIS dataset: https://github.com/yvchen/JointSLU; https://www.kaggle.com/siddhadev/atis-dataset-clean/home

The task is to classify the intent of a sentence. The dataset is split into train, validation and test sets. **Use the provided splits** to train and evaluate your models.


In [7]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/IntentDetection/atis.train.csv
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/IntentDetection/atis.dev.csv
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/IntentDetection/atis.test.csv

--2026-11-02 15:45:48--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/IntentDetection/atis.train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 838864 (819K) [text/plain]
Saving to: ‘atis.train.csv’

atis.train.csv      100%[===================>] 819.20K  --.-KB/s    in 0.05s   

2026-11-02 15:45:49 (17.3 MB/s) - ‘atis.train.csv’ saved [838864/838864]

--2026-11-02 15:45:49--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/IntentDetection/atis.dev.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, await

### **Question 5: Two-step classification model**

Train a classification model to identify the intent from a given sentence. The model is required to leverage on pre-trained BERT model to generate sentence embeddings (important: **no fine-tuning**). The model is required to use the embeddings to perform classification.

Once extracted the embeddings, you can use any classifier you want. For example, you can use a linear classifier (e.g., Logistic Regression) or a neural network (e.g., MLP). For your convenience, you can use the `sklearn` library for training the classifier (https://scikit-learn.org/stable/supervised_learning.html).

![https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P4/IntentDetection/no_finetuning.png?raw=true](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P4/IntentDetection/no_finetuning.png?raw=true)


Assess the performance of the trained model (the model on top of BERT) on the test set by using the **classification accuracy**, **precision**, **recall** and **F1-score**. You can use the `sklearn` library for computing the metrics (https://scikit-learn.org/stable/api/sklearn.metrics.html).


Note: you can use the `sentence-transformers` library to generate sentence embeddings (https://www.sbert.net/docs/pretrained_models.html).

In [ ]:
%%capture
!pip install sentence-transformers
!pip install sklearn

In [ ]:
# Required imports:

import pandas as pd
import numpy as np

In [ ]:
df_train = pd.read_csv('atis.train.csv', index_col='id')
df_val = pd.read_csv('atis.dev.csv',index_col='id')
df_test = pd.read_csv('atis.test.csv', index_col='id')

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Generate embedding using pre-trained encoder (no fine-tuning)
df_train['embs'] = df_train['tokens'].apply(model.encode)

df_test['embs'] = df_test['tokens'].apply(model.encode)

df_val['embs'] = df_val['tokens'].apply(model.encode)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# 3. Train a logistic regressor to predict labels (can take embedding vector as input directly)

from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression()

# Note: LogisticRegressor requires a true (m,n) numpy array.
# In our case we have a (m,) array containing (n,) objects.


classifier.fit(np.vstack(np.vstack(df_train['embs'].values)),df_train['intent'])
preds = classifier.predict(np.vstack(df_val['embs'].values))

In [ ]:
from sklearn.metrics import f1_score

print(f1_score(df_val['intent'],preds, average='weighted'))

0.910877068083422


In [ ]:
test_preds = classifier.predict(np.vstack(np.vstack(df_test['embs'].values)))
print(f1_score(df_test['intent'],test_preds, average='weighted'))

0.8885639042067649


### **Question 6: Fine-tuning end-to-end classification model**

Another approach is to fine-tune the BERT model for the classification task. A classification head is added on top of the pre-trained BERT model. The classification head is trained end-to-end with the BERT model.
This approach is more effective than the previous one because the model is trained end-to-end. However, the model requires more training time and resources.

Train a new BERT model for the task of [sequence classification](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForSequenceClassification) (include BERT fine-tuning).  

![https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P4/IntentDetection/finetuning.png?raw=true](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P4/IntentDetection/finetuning.png?raw=true)

Assess the performance of the generated model by using the same metrics used in the previous question.

Which model has better performance? Why?

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [9]:
# ----------------------------------------------------------------------
# 1. Configuration & Data Paths
# ----------------------------------------------------------------------
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 17

# Adjust column names according to your CSV structure
TEXT_COLUMN = "tokens"      # e.g., 'utterance' or 'text'
LABEL_COLUMN = "intent"   # e.g., 'intent' or 'label'

data_files = {
    "train": "atis.train.csv",
    "validation": "atis.dev.csv",
    "test": "atis.test.csv"
}

In [ ]:
len(np.unique(dataset['validation']['intent']))

17

### 2. Loading Datasets

We load datasets using HuggingFace's `load_dataset()` function from `dataset` library (python module).


**Why?**

Unlike standard Python tools like pandas.read_csv() (which loads the entire dataset directly into RAM), load_dataset() builds on *Apache Arrow*:

Datasets are memory-mapped from disk.

**You can work with datasets that are tens or hundreds of gigabytes in size without running out of RAM (Out Of Memory / OOM errors).**

Loading is nearly instantaneous because data is read dynamically from disk as batches are requested.

In [ ]:
# ----------------------------------------------------------------------
# 2. Load Dataset & Encode String Labels to Integers
# ----------------------------------------------------------------------
dataset = load_dataset("csv", data_files=data_files)

print(dataset)

# Fit LabelEncoder on the training set labels
label_encoder = LabelEncoder()
label_encoder.fit(dataset["train"][LABEL_COLUMN])

# Verify label count matches expected number
unique_labels_found = len(label_encoder.classes_)
print(f"Found {unique_labels_found} unique labels in training data.")
assert unique_labels_found == NUM_LABELS, f"Expected {NUM_LABELS} labels, but found {unique_labels_found}!"

# Create id2label and label2id mappings for the model configuration
id2label = {i: label for i, label in enumerate(label_encoder.classes_)} # creates number : label dictionary
label2id = {label: i for i, label in enumerate(label_encoder.classes_)} # crates label : number dictionary

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 4274
    })
    validation: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 572
    })
    test: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 586
    })
})
Found 17 unique labels in training data.


In [ ]:
# Function to transform string labels into numeric label IDs
def encode_labels(example):
    # Fallback to handling unseen labels during test/val if any exist
    try:
        example["label"] = int(label_encoder.transform([example[LABEL_COLUMN]])[0])
    except ValueError:
        example["label"] = 0  # Default fallback ID if label was unobserved in train
    return example

dataset = dataset.map(encode_labels)


'''


In that specific snippet, .map() operates on every individual split of your DatasetDict,
processing one row (example) at a time.

It's equivalent to this:

dataset["train"] = dataset["train"].map(encode_labels)
dataset["validation"] = dataset["validation"].map(encode_labels)
dataset["test"] = dataset["test"].map(encode_labels)


'''

Map:   0%|          | 0/4274 [00:00<?, ? examples/s]

Map:   0%|          | 0/572 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

'\n\n\nIn that specific snippet, .map() operates on every individual split of your DatasetDict, \nprocessing one row (example) at a time.\n\nIt\'s equivalent to this:\n\ndataset["train"] = dataset["train"].map(encode_labels)\ndataset["validation"] = dataset["validation"].map(encode_labels)\ndataset["test"] = dataset["test"].map(encode_labels)\n\n\n'

## 3. Tokenization of sentence



In [ ]:
# ----------------------------------------------------------------------
# 3. Tokenization (Without Static Padding)
# ----------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

'''

Loads the exact tokenizer vocabulary, byte-pair/WordPiece rules, and special tokens
([CLS], [SEP], [PAD], [UNK]) that were used when MODEL_NAME (e.g., bert-base-uncased)
was pre-trained.

'''

def preprocess_function(examples):
  return tokenizer(examples[TEXT_COLUMN], truncation=True, max_length=512)

'''

Truncate inputs: each sequence is truncated when exceding length of 512 tokens;
Since no padding strategy is specified, at this stage we might have sequences of
32 tokens living along sequences of length 64.

DataCollatorWithPadding will handle dynamic padding per batch


'''

tokenized_datasets = dataset.map(preprocess_function, batched=True)

'''

batched=True allows to pass inputs to map() function in chuncks, allowing multi-threading,
avoiding slow looping python implementation.

At this stage tokens gain id's and a mask (with all bits set to 1 since no padding is performed)

'''

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

'''

  This object is handed to PyTorch DataLoader or HuggingFace Trainer to handle dynamic padding.

'''

Map:   0%|          | 0/4274 [00:00<?, ? examples/s]

Map:   0%|          | 0/572 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

In [ ]:
# ----------------------------------------------------------------------
# 4. Load Pre-trained BERT Model with Custom Label Mappings
# ----------------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<u>Note</u>:

Why the necessity to specify id-to-labels mapping functions?

Neural networks operate strictly on numerical tensors—they compute linear projections, matrix multiplications,
dot-product attention, and softmax over continuous numerical values. They have no concept of strings or human-readable
names like "atis_flight" or "atis_airfare".

The output layer of BertForSequenceClassification is a standard linear
layer:

$$\mathbf{z} = \mathbf{W}\mathbf{h}_{\text{[CLS]}} + \mathbf{b}, \quad \mathbf{z} \in \mathbb{R}^{\text{num\_labels}}$$

In [ ]:
# ----------------------------------------------------------------------
# 5. Define Evaluation Metrics (Accuracy, Weighted F1, Macro F1)
# ----------------------------------------------------------------------
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

'''

logits: A 2D NumPy array of raw unnormalized model predictions
with shape (num_samples, num_labels). For 17 classes,
each row has 17 floating-point values.

labels: A 1D NumPy array of ground-truth integer IDs with shape (num_samples,).

'''
    predictions = np.argmax(logits, axis=-1)


    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_weighted = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {
        "accuracy": acc,
        "f1_weighted": f1_weighted,
        "f1_macro": f1_macro
    }

<u>Note</u>:

The resulting vector $\mathbf{z} = [z_1, z_2, \dots, z_C]$ is the logit vector.

**Key characteristics of logits**:

- Range: They can be any real number: $(-\infty, +\infty)$. They can be negative, zero, or positive.
- Not Probabilities: They do not sum to 1.
- Relative Confidence: A higher value $z_i$ indicates that the network found stronger activation/evidence for class $i$ relative to other classes.

Example: suppose $\mathbf{z} = [2.5, 1.0, -0.5]$

by applying the *SoftMax* function $$P(y = i \mid \mathbf{x}) = \sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$
to each vector entry, we obtain:

$P(\text{flight}) = \frac{12.1825}{15.5073} \approx 0.7856$ (78.56%)

$P(\text{airfare}) = \frac{2.7183}{15.5073} \approx 0.1753$ (17.53%)

$P(\text{ground\_service}) = \frac{0.6065}{15.5073} \approx 0.0391$ (3.91%)

**BUT** since the *SoftMax* function is monotonic, what we really need is just the highest logit (actually, its position therefore the class is points to).


In [ ]:
# ----------------------------------------------------------------------
# 6. Training Setup
# ----------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./bert_intent_classification",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    fp16=True,  # Set to False if training without GPU acceleration
    logging_steps=50, # Prints the running training loss to standard output and logging backends every 50 optimizer steps.
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# Check if CUDA (NVIDIA GPU) is available

import torch

is_available = torch.cuda.is_available()
print(f"CUDA Available: {is_available}")

if is_available:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    total_memory = torch.cuda.get_device_properties(current_device).total_memory / (1024 ** 3)

    print(f"Device Count: {device_count}")
    print(f"Current GPU: {device_name}")
    print(f"Total VRAM: {total_memory:.2f} GB")
else:
    print("No GPU detected by PyTorch. Code will fall back to CPU.")

CUDA Available: True
Device Count: 1
Current GPU: Tesla T4
Total VRAM: 14.56 GB


In [ ]:
# ----------------------------------------------------------------------
# 7. Execute Training & Evaluate on Held-out Test Set
# ----------------------------------------------------------------------
print("Starting BERT fine-tuning...")
trainer.train()

print("\n--- Final Evaluation on Validation Set ---")
eval_results = trainer.evaluate(tokenized_datasets["validation"])
print(eval_results)

print("\n--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

Starting BERT fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.302150,0.218929,0.958042,0.943385,0.458442
2,0.117218,0.124243,0.975524,0.969292,0.666977
3,0.080113,0.093979,0.984266,0.981658,0.784416


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


--- Final Evaluation on Validation Set ---


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted,F1 Macro
0.080113,0.093979,3,0.984266,0.981658,0.784416


{'eval_loss': 0.09397869557142258, 'eval_accuracy': 0.9842657342657343, 'eval_f1_weighted': 0.9816575288997048, 'eval_f1_macro': 0.7844161408961933}

--- Final Evaluation on Test Set ---


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted,F1 Macro
0.080113,0.107995,3,0.982935,0.979875,0.839621


{'eval_loss': 0.10799539089202881, 'eval_accuracy': 0.9829351535836177, 'eval_f1_weighted': 0.9798749397499958, 'eval_f1_macro': 0.8396205683919646}


In [ ]:
# ----------------------------------------------------------------------
# 8. Save Artifacts
# ----------------------------------------------------------------------
model.save_pretrained("./final_bert_intent_model")
tokenizer.save_pretrained("./final_bert_intent_model")
print("\nModel saved successfully to ./final_bert_intent_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully to ./final_bert_intent_model


### **Question 7: Intent Detection using Large Language Models**
**Credits:** Giuseppe Gallipoli

#### Introduction
[Large Language Models](https://en.wikipedia.org/wiki/Large_language_model) (LLMs) are a type of deep learning model capable of language generation. These models are built on deep learning architectures, primarily using neural networks, and are trained on massive amounts of text data. LLMs generally leverage the *Transformer* architecture, which allows them to process language in context, capturing complex relationships between words and concepts.

Large Language Models have demonstrated excellent capabilities across a wide variety of tasks, making them versatile models which can be applied in diverse scenarios and use cases. Although they are more typically used for *generative* tasks (e.g., text generation, text summarization, open-ended Question Answering), they can also be employed in *discriminative* tasks.

In this practice, we will use a Large Language Model to address an intent detection task. Rather than using a pre-trained encoder-only model without (Question 5) or with fine-tuning (Question 6), we will ask the LLM to classify the intent given a sentence of interest.
<br><br>
For now, we will use the [Phi-4-mini-instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct) 3.8B model, i.e., `microsoft/Phi-4-mini-instruct`.

#### LLM Prompting
To interact with LLMs, we need to define a **prompt**, which is a piece of text containing the instruction or question we want to give or ask the model. As we saw in Practice 1 (Exercise 11), a prompt can both include only the instruction/question for the LLM but also additional information (i.e., *context*) which can be exploited by the model to generate the answer.

For now, we will ask the model to classify the intent of a given sentence <u>without providing</u> any additional context. Please note that, since we want to (potentially) limit the choice of the LLM to a predefined list of intents (i.e., the set of labels of our dataset), we will also provide this list to the model.\


*Example of prompt*:\
Which is the intent of the following sentence?\
Choose among: label1, label2, ...
<br><br>

**Hint**: For those data instances whose intent is the concatenation of two intents (i.e., `atis_flight#atis_airfare`), consider only the first one.

<u>Suggestion</u>: To increase speed, switch to a GPU runtime. You can do this by clicking on Runtime → Change runtime type → Hardware accelerator → Select T4 GPU.\
If you encounter an `OutOfMemoryError`, try restarting the session by clicking on Runtime → Restart session.

## <u>Note</u>: why choosing an LLM rather than a fine-tuned model like BERT: a comparison

| Dimension | Fine-Tuned Encoder (e.g., BERT) | Large Language Model (Prompted / In-Context) |
| :--- | :--- | :--- |
| **Labeled Data Requirement** | High (hundreds to thousands of labeled samples per class) | Zero to Minimal (zero-shot descriptions or 2–5 examples per class) |
| **Inference Latency** | **~2–10 ms** (on GPU/CPU; easily quantized with ONNX/TensorRT) | **~150–1000+ ms** (network overhead, auto-regressive decoding) |
| **Operational & Compute Cost** | Very low (can run on cheap CPUs or small, shared GPUs) | High (API token costs or dedicated high-VRAM GPU clusters) |
| **Schema Flexibility** | Rigid (adding a new intent requires retraining the classification head) | High (add a new intent simply by editing the system prompt) |
| **Output Capability** | Single discrete class ID (fixed closed-set classification) | Complex structured outputs (multi-intent, slot filling, JSON, reasoning) |
| **Interpretability / Control** | Deterministic logits, clear probabilities, calibration curves | Potential hallucination, non-deterministic phrasing unless constrained |


In [1]:
!pip install evaluate
!pip install transformers
!pip install accelerate

In [5]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [10]:
model_id = "microsoft/Phi-4-mini-instruct"

# 1. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

# 2. Candidate intent labels (e.g., your 17 ATIS classes)
INTENT_LABELS = [
    "atis_flight", "atis_airfare", "atis_ground_service", "atis_airline",
    "atis_abbreviation", "atis_aircraft", "atis_flight_time", "atis_quantity",
    "atis_city", "atis_distance", "atis_airport", "atis_capacity",
    "atis_flight_no", "atis_ground_fare", "atis_meal", "atis_restriction", "atis_cheapest"
]

# Build the string of all possible lables.

labels_str = ", ".join(INTENT_LABELS)

# 3. Define the instruction to be submitted to the LLM

system_instruction = (
    "You are a strict intent classification engine. "
    "Classify the user's sentence into EXACTLY ONE of the following valid intent classes:\n"
    f"{labels_str}\n\n"
    "Respond with ONLY the exact intent name and nothing else."
)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [ ]:
# 4. Define the classifaction function:

def classify_intent(input_sequence: str ) -> str:

  '''

  The following structure is the standard Chat Message format (originally popularized by the OpenAI Chat Completions API
  and now the universal standard in Hugging Face, vLLM, and Ollama).
  It represents a conversation as a list of message dictionaries, where each dictionary has two required keys:

  "role": Identifies who is speaking ("system", "user", or "assistant").
  "content": The actual text string of that turn.

  What Each Role Means:

  {"role": "system", ...}:
  Purpose: Sets the global behavior, constraints, tone, and operational boundaries of the model.
  In your classification snippet, it sets the model's persona, provides the strict list of 17 candidate intent classes,
  and enforces the rule: "Output ONLY the label name."

  {"role": "user", ...}:
  Purpose: The prompt or query submitted by a human (or caller).
  In your case, it supplies the test sentence to be classified.

  {"role": "assistant", ...} (used when adding few-shot examples or history):
  Purpose: Represents responses generated by the AI.

  When teaching the model via few-shot prompting,
  you insert paired user and assistant messages to show the exact output format you expect.

  '''
  messages = [
    {"role": "system", "content": system_instruction},
    # (Optional: include 2-3 few-shot examples here to anchor formatting)
    {"role": "user", "content": f"Sentence: \"{input_sequence}\"\nIntent:"}
  ]

  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  # transforms dictionaries in a full prompt, python string
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # (pt stands for PyTorch Tensor)
  # tokenizes the prompt and moves it to gpu/cpu, where the model resides.

  with torch.no_grad(): # disables torch's automatic differentiation engine
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=10,  # Intent names are short; keep this small to prevent rambling (generating additional non-sense)
            temperature=0.0,    # Greedy decoding for deterministic classification
            do_sample=False,
            # disables random multinomial sampling, returning the class that maximizes the log likelihood
            # this behavior is not recommended in text generation per se since produces deterministic
            # non-creative outputs
            pad_token_id=tokenizer.eos_token_id, # assigns EOS token at the end, ending generation
        )

  input_len = inputs["input_ids"].shape[1]
  # model returns input tokens followed by fresh-new ones. We want to output just the new-ones
  prediction = tokenizer.decode(generated_ids[0][input_len:], skip_special_tokens=True).strip()
  return prediction

In [11]:
# Adjust column names according to your CSV structure
TEXT_COLUMN = "tokens"      # e.g., 'utterance' or 'text'
LABEL_COLUMN = "intent"   # e.g., 'intent' or 'label'
NUM_LABELS = 17

data_files = {
    "train": "atis.train.csv",
    "validation": "atis.dev.csv",
    "test": "atis.test.csv"
}

# ----------------------------------------------------------------------
# Load Dataset & Encode String Labels to Integers
# ----------------------------------------------------------------------
dataset = load_dataset("csv", data_files=data_files)

print(dataset)

# Fit LabelEncoder on the training set labels
label_encoder = LabelEncoder()
label_encoder.fit(dataset["train"][LABEL_COLUMN])

# Verify label count matches expected number
unique_labels_found = len(label_encoder.classes_)
print(f"Found {unique_labels_found} unique labels in training data.")
assert unique_labels_found == NUM_LABELS, f"Expected {NUM_LABELS} labels, but found {unique_labels_found}!"

# Create id2label and label2id mappings for the model configuration
id2label = {i: label for i, label in enumerate(label_encoder.classes_)} # creates number : label dictionary
label2id = {label: i for i, label in enumerate(label_encoder.classes_)} # crates label : number dictionary

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 4274
    })
    validation: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 572
    })
    test: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 586
    })
})
Found 17 unique labels in training data.


In [ ]:
# 3. Run evaluation across the test set

from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score

candidate_labels = label_encoder.classes_

test_data = dataset["test"]
y_true = []
y_pred = []

print(f"Running zero-shot inference on {len(test_data)} test samples...")
for sample in tqdm(test_data):
    utterance = sample[TEXT_COLUMN]
    true_label = sample[LABEL_COLUMN]

    pred_label = classify_intent(utterance)

    # Handle rare cases where the model outputs unexpected text or extra spaces
    if pred_label not in candidate_labels:
        # Fallback: check if any valid label is a substring of the output
        matched = [lbl for lbl in candidate_labels if lbl in pred_label]
        pred_label = matched[0] if matched else "unrecognized"

    y_true.append(true_label)
    y_pred.append(pred_label)

# 4. Compute Metrics
acc = accuracy_score(y_true, y_pred)
print(f"\nZero-shot Accuracy: {acc:.4f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

Running zero-shot inference on 586 test samples...


  0%|          | 0/586 [00:00<?, ?it/s]


Zero-shot Accuracy: 0.3754

                          precision    recall  f1-score   support

       atis_abbreviation       0.76      0.81      0.79        16
           atis_aircraft       0.86      0.75      0.80         8
            atis_airfare       0.92      0.85      0.88        54
            atis_airline       0.57      0.44      0.50        18
            atis_airport       0.17      1.00      0.30         4
           atis_capacity       1.00      0.50      0.67         4
               atis_city       0.00      0.00      0.00         3
           atis_distance       0.50      0.67      0.57         3
             atis_flight       0.94      0.24      0.39       424
atis_flight#atis_airfare       0.00      0.00      0.00         3
          atis_flight_no       0.14      0.50      0.22         2
        atis_flight_time       0.02      1.00      0.04         7
        atis_ground_fare       0.75      0.75      0.75         4
     atis_ground_service       0.96      0.86 

#### Few-Shot Learning
When we provide the LLM with additional context to be leveraged for generating an answer, this is known as *in-context learning* (ICL).\
A common technique to perform ICL involves including one or more **additional examples** of questions (or instructions) and their expected answers in the prompt.\
This can be useful for several reasons: telling the model the required output format, both in terms of structure and style, making the model reason about the provided examples to better adapt to the (new) task, tailoring model's responses to a user's specific needs, ...\
All of this is done directly at inference, without the need of further training the model, simply by leveraging the reasoning and generalization capabilities of LLMs.\
When no examples are provided, as in the previous point of the exercise, we talk about **zero-shot** learning. Conversely, when we supply additional input-output examples in the model's prompt, we talk about **few-shot** learning or $k$-shot learning, where $k$ represents the number of ICL examples.\
Examples can be selected according to different strategies, either randomly or in a more clever way, but please note that they <u>must be chosen from the training set</u> to avoid *data leakage*.
<br><br>

Now, try to implement few-shot learning by modifying the previous point of the exercise so that the prompt can include $k$ additional examples.

In [41]:
few_shots_list = []
seen_intents = set()

# Shuffle first to ensure random representatives for each intent
shuffled_ds = dataset['validation'].shuffle(seed=42)

for val in shuffled_ds:
    intent = val['intent']
    if intent not in seen_intents:
        seen_intents.add(intent)
        sentence = val['tokens'][4:-3]
        few_shots_list.append((f'Sentence: "{sentence}"\nIntent:', intent))

print(few_shots_list)
print(f"Total unique intent samples collected: {len(few_shots_list)}")

[('Sentence: "what is the cost for a one way trip from pittsburgh to atlanta "\nIntent:', 'atis_airfare'), ('Sentence: "round trip flights from salt lake city to cincinnati nonstop flights if possible "\nIntent:', 'atis_flight'), ('Sentence: "list the number of flights leaving boston for dallas fort worth before 9 am in the morning "\nIntent:', 'atis_quantity'), ('Sentence: "what airlines serve denver "\nIntent:', 'atis_airline'), ('Sentence: "what is the seating capacity on the aircraft m80 "\nIntent:', 'atis_capacity'), ('Sentence: "please list the rental car information in pittsburgh "\nIntent:', 'atis_ground_service'), ('Sentence: "list departure times from denver to philadelphia which are later than 10 o\'clock and earlier than 2 pm "\nIntent:', 'atis_flight_time'), ('Sentence: "i\'m going to leave philadelphia and i want to go to san francisco and i want to fly first class american and i want a stop in dallas can you please tell me what type of aircraft you will be flying "\nInte

In [42]:
def classify_intent2(input_sequence: str ) -> str:

    messages = [{"role": "system", "content": system_instruction}]

    #few_shots_examples
    for user_query, label in few_shots_list:
      messages.append({"role": "user", "content": user_query})
      messages.append({"role": "assistant", "content": label})

    # Current target
    messages.append(
      {"role": "user", "content": f'Sentence: "{input_sequence}"\nIntent:'}
    )


    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # transforms dictionaries in a full prompt, python string
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # (pt stands for PyTorch Tensor)
    # tokenizes the prompt and moves it to gpu/cpu, where the model resides.

    with torch.no_grad(): # disables torch's automatic differentiation engine
          generated_ids = model.generate(
              **inputs,
              max_new_tokens=10,  # Intent names are short; keep this small to prevent rambling (generating additional non-sense)
              temperature=0.0,    # Greedy decoding for deterministic classification
              do_sample=False,
              # disables random multinomial sampling, returning the class that maximizes the log likelihood
              # this behavior is not recommended in text generation per se since produces deterministic
              # non-creative outputs
              pad_token_id=tokenizer.eos_token_id, # assigns EOS token at the end, ending generation
          )

    input_len = inputs["input_ids"].shape[1]
    # model returns input tokens followed by fresh-new ones. We want to output just the new-ones
    prediction = tokenizer.decode(generated_ids[0][input_len:], skip_special_tokens=True).strip()
    return prediction

In [43]:
# Adjust column names according to your CSV structure
TEXT_COLUMN = "tokens"      # e.g., 'utterance' or 'text'
LABEL_COLUMN = "intent"   # e.g., 'intent' or 'label'
NUM_LABELS = 17

data_files = {
    "train": "atis.train.csv",
    "validation": "atis.dev.csv",
    "test": "atis.test.csv"
}

# ----------------------------------------------------------------------
# Load Dataset & Encode String Labels to Integers
# ----------------------------------------------------------------------
dataset = load_dataset("csv", data_files=data_files)

print(dataset)

# Fit LabelEncoder on the training set labels
label_encoder = LabelEncoder()
label_encoder.fit(dataset["train"][LABEL_COLUMN])

# Verify label count matches expected number
unique_labels_found = len(label_encoder.classes_)
print(f"Found {unique_labels_found} unique labels in training data.")
assert unique_labels_found == NUM_LABELS, f"Expected {NUM_LABELS} labels, but found {unique_labels_found}!"

# Create id2label and label2id mappings for the model configuration
id2label = {i: label for i, label in enumerate(label_encoder.classes_)} # creates number : label dictionary
label2id = {label: i for i, label in enumerate(label_encoder.classes_)} # crates label : number dictionary

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 4274
    })
    validation: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 572
    })
    test: Dataset({
        features: ['id', 'tokens', 'slots', 'intent'],
        num_rows: 586
    })
})
Found 17 unique labels in training data.


In [44]:
# 3. Run evaluation across the test set

from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score

candidate_labels = label_encoder.classes_

test_data = dataset["test"]
y_true = []
y_pred = []

print(f"Running zero-shot inference on {len(test_data)} test samples...")
for sample in tqdm(test_data):
    utterance = sample[TEXT_COLUMN]
    true_label = sample[LABEL_COLUMN]

    pred_label = classify_intent2(utterance)

    # Handle rare cases where the model outputs unexpected text or extra spaces
    if pred_label not in candidate_labels:
        # Fallback: check if any valid label is a substring of the output
        matched = [lbl for lbl in candidate_labels if lbl in pred_label]
        pred_label = matched[0] if matched else "unrecognized"

    y_true.append(true_label)
    y_pred.append(pred_label)

# 4. Compute Metrics
acc = accuracy_score(y_true, y_pred)
print(f"\nZero-shot Accuracy: {acc:.4f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

Running zero-shot inference on 586 test samples...


  0%|          | 0/586 [00:00<?, ?it/s]


Zero-shot Accuracy: 0.2287

                          precision    recall  f1-score   support

       atis_abbreviation       0.61      0.69      0.65        16
           atis_aircraft       0.80      1.00      0.89         8
            atis_airfare       0.78      0.87      0.82        54
            atis_airline       0.24      0.44      0.31        18
            atis_airport       0.07      1.00      0.12         4
           atis_capacity       1.00      1.00      1.00         4
               atis_city       0.33      0.33      0.33         3
           atis_distance       1.00      0.67      0.80         3
             atis_flight       1.00      0.02      0.04       424
atis_flight#atis_airfare       0.33      0.67      0.44         3
          atis_flight_no       0.01      1.00      0.02         2
        atis_flight_time       0.04      1.00      0.08         7
        atis_ground_fare       0.50      0.25      0.33         4
     atis_ground_service       1.00      0.93 

Try with different $k$ values, e.g., $k \in [1, 3, 5, 10]$, and see if and how the model's performance changes.

In [ ]:
# your code here

Until now, we have used the [Phi-4-mini-instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct) 3.8B model, i.e., `microsoft/Phi-4-mini-instruct`, but there are plenty of LLMs!\
Try another model of your choice, e.g., [Mistral-Instruct](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) 7B `mistralai/Mistral-7B-Instruct-v0.3`. Remember to always keep in mind the GPU memory constraints you have.
<br><br>

**Hint**: To download and use certain models, it may be needed to request access to them (e.g., [Llama3.2-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) 3B). This can be done from the Hugging Face model page after logging in. Once access is granted, you need to create your personal access token (go to your HF profile, Settings → Access Tokens → Create new token) and authenticate using it when downloading both the model and the tokenizer.\
To authenticate, you can either use the additional `token` parameter in the `from_pretrained()` method or via the [HF Command Line Interface](https://huggingface.co/docs/huggingface_hub/guides/cli).
```
model_tag = 'MODEL_TAG'
HF_TOKEN = 'MY_HF_TOKEN'

model = AutoModelForCausalLM.from_pretrained(model_tag, torch_dtype=torch.float16, token=HF_TOKEN, device_map=device)
tokenizer = AutoTokenizer.from_pretrained(model_tag, token=HF_TOKEN)
```

In [ ]:
# your code here

After analyzing the performance of the different settings and/or models, here you can find some questions to reason about the results:
- What is the performance in the zero-shot learning setting? (i.e., $k=0$)
- How does it change when providing one additional example? (i.e., $k=1$)
- What happens when increasing the number of ICL examples?
- If you tested additional models, which one performs best in the zero-shot and few-shot learning settings?
- What challenges or limitations did you observe?